# 04 · Two loops: Hermes-style vs governed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AreevAI/dejadb/blob/main/examples/colab/04_hermes_vs_governed.ipynb)

*Segments: "What is Hermes and how it creates and reuses skills" + the
Hermes–DejaDB comparison.*

[Hermes Agent](https://github.com/nousresearch/hermes-agent) (Nous Research,
MIT) is the best-known open self-improving agent: a self-hosted daemon that
reflects after each turn and writes what it learned — reusable **skills** and
consolidated memory — as markdown files loaded into future prompts. Great
runtime, genuinely good skills idea. The part enterprises trip over is one
default: **`memory.write_approval` and `skills.write_approval` are off** — the
agent edits its own behavior, unreviewed.

Scene: an online retailer's refund agent, during a flash sale.

In [1]:
# dejadb 1.0.5 is on PyPI; `dejadb.helpers` ships inside the wheel.
%pip install -q "dejadb>=1.0.5" matplotlib

from dejadb.helpers import *
import dejadb, json, pathlib
print("dejadb", dejadb.__version__)

dejadb 1.0.3


## Loop A · reflection with the gate off (simulated in 12 lines)

During the flash-sale launch, `refund_processor` fails twice. Post-turn
reflection distills a takeaway and writes it straight into the skills file:

In [2]:
SKILLS = pathlib.Path("skills.md")
SKILLS.write_text("# learned skills\n")

def reflect(takeaway):                       # write_approval: off
    SKILLS.write_text(SKILLS.read_text() + f"- {takeaway}\n")

# the episode: flash sale live + two refund failures. correlation, meet causation:
reflect("avoid refund_processor while a flash sale is running")
print(SKILLS.read_text())

# learned skills
- avoid refund_processor while a flash sale is running



Three problems, none visible to anyone:

1. **Wrong** — two failures during an unrelated gateway restart became policy.
   The agent will now sit on customer refunds during every big sale — precisely
   when refund volume peaks.
2. **No evidence** — the file states a conclusion; nobody can see what it was
   based on, or who wrote it.
3. **No history** — correcting it means editing the file, and then even the
   mistake disappears:

In [3]:
SKILLS.write_text("# learned skills\n")      # "fixed" — and untraceable
print(SKILLS.read_text())
print("what did the agent believe during the sale? no record.")

# learned skills

what did the agent believe during the sale? no record.


## Loop B · the same experience, governed

Same events, recorded as **evidence** instead of conclusions:

In [4]:
db = fresh("refunds.db", ns="ops", actor="agent:refund-bot")

db.add_fact("storefront", "campaign", "flash sale live")
db.record_tool_call("refund_processor", "503 unavailable (call 1)", is_error=True, thread="sale")
db.record_tool_call("refund_processor", "503 unavailable (call 2)", is_error=True, thread="sale")

run = json.loads(db.waiser_run(full_sweep=True, model=auto_model()))
hunches = [r for r in recs(db) if r["analyzer"].startswith("waiser.llm")]
for r in hunches:
    print("the model drafted a hunch, and it QUEUED instead of becoming behavior:")
    print("   ", r["summary"])
    db.dismiss_recommendation(r["hash"],
        "correlation, not cause — payments team confirmed a gateway restart")
if not hunches:
    print("two failures: nothing cleared the evidence bar — no lesson, no drift")

two failures: nothing cleared the evidence bar — no lesson, no drift


That cell is the whole argument. Deterministic analyzers won't invent the
flash-sale correlation (proposals are evidence-bound and threshold-gated), and
if the **LLM** drafts that very hunch, it lands in a review queue with a
dismissal reason on the record — not in the agent's behavior. When the evidence
*does* clear the bar, adoption is one reviewed step:

In [5]:
db.record_tool_call("refund_processor", "503 unavailable (call 3)", is_error=True, thread="sale2")
db.waiser_run(full_sweep=True)
rec = [r for r in show_recs(db) if "tool_failure" in r["analyzer"]][0]
db.apply_recommendation(rec["hash"],
    because="payments confirmed rolling restart — retry refunds with backoff meanwhile")

  [medium] [reversible ] Tool "refund_processor" failed 3 times (100% of calls): # unavailable (call #)


'{"hash":"eaeec3cd58cdd6fe44a60b3157128d6e2ece6c96e3a3582d8fa9ebfe7815b073","rollbackable":true}'

> **Where the LLM fits.** `auto_model()` picks up an `OPENROUTER_API_KEY` from
> Colab's Secrets panel (the key icon in the left sidebar) or the
> environment. With a key, the same sweep adds an LLM **discovery** pass whose
> drafts must survive GROUND (does the cited evidence exist?) and VERIFY (an
> independent soundness check, 0.75 confidence floor) before they may even
> *queue* — the model never gets write access. Without a key, the deterministic
> analyzers still run: the floor is always keyless.

## The comparison that matters

| The learning loop | Hermes-style (gate off) | Governed (DejaDB + Waiser) |
|---|---|---|
| Write path | reflection → file, immediate | evidence → proposal → **review** → apply |
| Evidence | conclusion only | every proposal carries bounded evidence hashes |
| Wrong lesson | adopted silently | thresholds + GROUND→VERIFY + human approval |
| Correction | edit the file — history gone | supersede / rollback — history immutable |
| Measurement | — | baseline → current per lesson; regressions file their own revert |
| Audit | file mtime | actor + reason on every decision, content-addressed chain |
| Skills reuse | ✅ markdown skills, great DX | lessons as recallable facts (skill *synthesis* on the roadmap) |
| Agent runtime | ✅ full daemon, channels, cron | — bring your own (any framework / MCP / memory-tool) |

Honest bottom line: **complementary**. Hermes is a runtime whose learning loop
trusts the model; DejaDB + Waiser makes the loop reviewable — and could sit
underneath a Hermes-style runtime as its governed memory tier. If your agents
touch customers, money, or production, the governance rows are the ones your
auditors will ask about.

*(Design claims about Hermes are from its own docs — write-approval defaults,
file-based memory and skills; we compare designs, not popularity.)*